# 04 · Beyond 77 K: the DOE swing test and fine-tuning to 130 K
Reproduces report section 6.4:
* 160 K/5 bar GCMC on 23 MOFs, showing the 77 K capacity is a poor proxy for the DOE swing (`compare_160.py`).
* 130 K labels for 157 generated + 11 external MOFs, and four transfer methods (`finetune_130.py`).
* 130 K labels for 58 real library MOFs and the effect of adding them (`prep_real.py`, `finetune_130_real.py`).

**Runtime: about 5–7 hours on 4 cores** (665 runs). **Attach:** the `mofh2_pipeline` dataset; optionally stage 02/03 outputs.

### Settings
* `USE_REFERENCE_UPSTREAM`: fill results this notebook needs from earlier stages with the original project's runs (packaged in `reference_runs.tgz`) when you have not attached your own earlier notebook outputs. Lets every notebook run on its own.
* `RECOMPUTE_THIS_STAGE`: re-run this notebook's own simulations. Set `False` to only redo the analysis on the reference runs.
* `MAX_HOURS`: stop launching new simulations after this many hours (Kaggle sessions end at 12 h). Every queue is resumable: save the output, attach it to a new session, run again.

In [ ]:
import os, glob, subprocess
PKG = os.path.dirname(glob.glob('/kaggle/input/**/mofh2_pipeline/setup_env.sh', recursive=True)[0])
os.environ['WORK'] = WORK = '/kaggle/working/mofh2'
os.environ['NPROC'] = str(os.cpu_count())
USE_REFERENCE_UPSTREAM = True
RECOMPUTE_THIS_STAGE = True
MAX_HOURS = 11.0
STAGE_QUEUES = ['04_beyond77_130K']
print('package:', PKG, '| cores:', os.cpu_count())

### Setup: build RASPA2 and Zeo++, install packages (about 5–8 minutes)

In [ ]:
!bash $PKG/setup_env.sh

### Restore results from earlier stages

In [ ]:
%cd $WORK
!python stage_io.py restore {'--reference' if USE_REFERENCE_UPSTREAM else ''}
if RECOMPUTE_THIS_STAGE:
    for q in STAGE_QUEUES:
        !python stage_io.py forget queues/{q}.txt

In [ ]:
!python featurise_t130.py

### Run simulations: `04_beyond77_130K` (665 runs)

In [ ]:
!bash rq_any.sh queues/04_beyond77_130K.txt {MAX_HOURS}

In [ ]:
!python compare_160.py

In [ ]:
!python finetune_130.py

In [ ]:
!python featurise_real.py && python finetune_130_real.py

### Compare with the original project and save outputs
The comparison prints mean and maximum differences between your reproduced numbers and the original ones. GCMC is stochastic, so expect small differences (typically under 1 g/L at 77 K/100 bar).

In [ ]:
!python compare_reference.py $PKG doe_swing_test.csv finetune_130_results.csv finetune_130_real_results.csv

In [ ]:
!python stage_io.py save stage04_outputs
# Attach /kaggle/working/stage04_outputs.tgz (via 'Save Version' output) as input to later notebooks.